### Treffsicherheit – Lösung Schritt für Schritt

*Jugendwettbewerb Informatik 2021, 3. Runde, Junioraufgabe 2 (= 40. Bundeswettbewerb Informatik, 1. Runde, Junioraufgabe 2)*

Vor jedem Treffen ihrer Clique schlägt Ada verschiedene Termine dafür im Internet vor. Jedes Mitglied kann für jeden dieser
Termine im Browser durch Auswahl einer Farbe angeben, ob der Termin ihm oder ihr sehr gut (grün), nur mäßig (gelb, ?) oder
überhaupt nicht (rot, X) gefällt. So entsteht eine Präferenztabelle. Mit deren Hilfe wird der passendste Termin bestimmt,
und der Weg für das nächste Treffen ist frei.

Besonders freut es Ada, wenn sich aus der Tabelle ein Termin ergibt, der so beliebt ist, dass keinem einzigen Mitglied ein
anderer Termin besser gefällt. Ansonsten möchte Ada wissen, wie viele Einträge in der Präferenztabelle wenigstens verändert
werden müssten, damit ein derartiger „allseits beliebter“ Termin entsteht. Für die Tabelle auf dem Aufgabenblatt ist diese Zahl 2:
Wenn Niklaus' Angabe für Termin 4 von gelb auf rot und Rószas Angabe für Termin 5 von grün auf gelb abgeändert werden, wird
Termin 6 zu einem allseits beliebten Termin.

#### Junioraufgabe 2

Hilf Ada und schreibe ein Programm, das eine Präferenztabelle einliest und berechnet und ausgibt, wie viele Einträge
wenigstens verändert werden müssten, damit ein allseits beliebter Termin entsteht. Lass auch diesen Termin ausgeben.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Die Tabelle auf dem Aufgabenblatt (`praeferenzen0.txt`)

6 Mitglieder (Ada, Nancy, Niklaus, Grace, Edsger, Rósza), 7 Termine. Die Zahlen bedeuten 0 = grün (sehr gut), 1 = gelb (mäßig),
2 = rot (überhaupt nicht):

```
6 7
0 0 0 0 0 0 0
1 0 0 1 1 0 0
2 2 2 1 2 2 2
2 1 1 1 2 1 2
0 1 2 2 1 0 0
1 2 1 2 0 1 1
```

#### Beispieldaten

Im Ordner `beispieldaten` stehen `praeferenzen0.txt` bis `praeferenzen5.txt`. In der ersten Zeile stehen die Anzahl `n` der Mitglieder
und die Anzahl `m` der Termine, dann folgt je Mitglied eine Zeile mit `m` Zahlen (0, 1 oder 2).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Tabelle als Liste von Listen:** `tabelle[i][t]` ist die Bewertung, die Mitglied `i` dem Termin `t` gibt.
- **Minimum suchen:** Die beste Bewertung eines Mitglieds ist der kleinste Wert in seiner Zeile. Der beste Termin ist der mit den
  wenigsten Konflikten, also wieder ein Minimum.
- **Zählen:** Für jeden Termin zählen wir die Mitglieder, bei denen ein anderer Termin besser bewertet ist (die „Konflikte“).
- **Vorberechnen:** Die beste Bewertung jedes Mitglieds berechnen wir einmal am Anfang, statt sie für jeden Termin neu zu suchen.
  Das ist ein typischer Trick, um ein Programm schneller zu machen.
- **Eine Aussage prüfen:** Als Kontrolle schreiben wir die Definition „allseits beliebt“ direkt als Funktion hin und testen damit unser Ergebnis.

**Nützliche Funktionen** (Bibliotheken braucht man keine)
- `min(liste)` liefert den kleinsten Wert einer Liste, `liste.index(wert)` die Position des ersten Vorkommens, `map(int, ...)` bzw. `int(...)`
  wandelt Texte in Zahlen um.

## Die Idee

Ein Termin ist **allseits beliebt**, wenn für **jedes** Mitglied gilt: Kein anderer Termin ist besser bewertet (kleinere Zahl = besser).
Mit anderen Worten: Jedes Mitglied bewertet diesen Termin so gut wie seinen besten Termin.

Wir betrachten einen Termin `t`. Ein Mitglied, für das `t` **schlechter** ist als seine beste Bewertung, ist ein **Konflikt**.
Um einen Konflikt zu beseitigen, muss man mindestens einen Eintrag ändern, und eine Änderung reicht auch: Man setzt die Bewertung des Termins `t` bei
diesem Mitglied auf seine beste Bewertung. Die minimale Anzahl an Änderungen, die den Termin `t` allseits beliebt macht, ist also die **Anzahl der Konflikte** von `t`.

Wir rechnen die Konflikte für alle Termine aus und nehmen den Termin mit den **wenigsten**. Das ist die Antwort.

## Schritt 1: Die Tabelle einlesen

Alle Zahlen der Datei werden mit `split()` gelesen. Die ersten beiden sind `n` und `m`, danach folgen `n · m` Bewertungen, Zeile für Zeile.
Wir bauen daraus eine Liste von `n` Zeilen mit je `m` Zahlen.

In [ ]:
def lies_tabelle(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zahlen = f.read().split()
    n = int(zahlen[0])                      # Anzahl der Mitglieder
    m = int(zahlen[1])                      # Anzahl der Termine
    tabelle = []
    position = 2
    for i in range(n):
        zeile = []
        for t in range(m):
            zeile.append(int(zahlen[position]))
            position = position + 1
        tabelle.append(zeile)
    return tabelle


tabelle = lies_tabelle('beispieldaten/praeferenzen0.txt')
for zeile in tabelle:
    print(zeile)

[0, 0, 0, 0, 0, 0, 0]
[1, 0, 0, 1, 1, 0, 0]
[2, 2, 2, 1, 2, 2, 2]
[2, 1, 1, 1, 2, 1, 2]
[0, 1, 2, 2, 1, 0, 0]
[1, 2, 1, 2, 0, 1, 1]


## Schritt 2: Die beste Bewertung jedes Mitglieds

Die beste Bewertung ist der kleinste Wert der Zeile (`0` ist besser als `1`, `1` besser als `2`). Wir berechnen sie für alle Mitglieder
einmal und merken uns die Werte in einer Liste `beste`.

In [ ]:
def beste_bewertungen(tabelle):
    beste = []
    for zeile in tabelle:
        beste.append(min(zeile))
    return beste


beste = beste_bewertungen(tabelle)
print(beste)

[0, 0, 1, 1, 0, 0]


## Schritt 3: Konflikte für einen Termin

Für den Termin `t` gehen wir alle Mitglieder durch. Ist die Bewertung von `t` **größer** (schlechter) als die beste Bewertung des Mitglieds,
ist das ein Konflikt.

In [ ]:
def konflikte(tabelle, beste, t):
    anzahl = 0
    for i in range(len(tabelle)):
        if tabelle[i][t] > beste[i]:
            anzahl = anzahl + 1
    return anzahl


for t in range(len(tabelle[0])):
    print(f'Termin {t + 1}: {konflikte(tabelle, beste, t)} Konflikte')

Termin 1: 4 Konflikte
Termin 2: 3 Konflikte
Termin 3: 3 Konflikte
Termin 4: 3 Konflikte
Termin 5: 4 Konflikte
Termin 6: 2 Konflikte
Termin 7: 3 Konflikte


Wir zählen die Termine ab 1 (so wie in der Aufgabe). Intern fangen die Listen bei 0 an, deshalb schreiben wir bei der Ausgabe `t + 1`.
Termin 6 hat die wenigsten Konflikte, nämlich 2, wie in der Aufgabenstellung angegeben.

## Schritt 4: Den besten Termin finden

Wir berechnen die Konflikte aller Termine und merken uns die kleinste Anzahl. Haben mehrere Termine gleich viele Konflikte, sind alle gleich gut.
Wir sammeln deshalb **alle** besten Termine und geben den ersten als Antwort an.

In [ ]:
def bester_termin(tabelle):
    beste = beste_bewertungen(tabelle)
    alle = []                               # Konflikte für jeden Termin
    for t in range(len(tabelle[0])):
        alle.append(konflikte(tabelle, beste, t))
    minimum = min(alle)
    termine = []
    for t in range(len(alle)):
        if alle[t] == minimum:
            termine.append(t)               # gezählt ab 0
    return minimum, termine


def bearbeite(dateiname):
    tabelle = lies_tabelle(dateiname)
    aenderungen, termine = bester_termin(tabelle)
    nummern = []
    for t in termine:
        nummern.append(str(t + 1))
    print(f'{dateiname}: {len(tabelle)} Mitglieder, {len(tabelle[0])} Termine')
    print(f'  Mindestens nötige Änderungen: {aenderungen}, bester Termin: {termine[0] + 1}')
    print(f'  (gleich gute Termine: {", ".join(nummern)})')


bearbeite('beispieldaten/praeferenzen0.txt')

beispieldaten/praeferenzen0.txt: 6 Mitglieder, 7 Termine
  Mindestens nötige Änderungen: 2, bester Termin: 6
  (gleich gute Termine: 6)


## Schritt 5: Alle Beispiele

In [ ]:
for nr in range(6):
    bearbeite(f'beispieldaten/praeferenzen{nr}.txt')

beispieldaten/praeferenzen0.txt: 6 Mitglieder, 7 Termine
  Mindestens nötige Änderungen: 2, bester Termin: 6
  (gleich gute Termine: 6)
beispieldaten/praeferenzen1.txt: 5 Mitglieder, 5 Termine
  Mindestens nötige Änderungen: 1, bester Termin: 2
  (gleich gute Termine: 2, 3)
beispieldaten/praeferenzen2.txt: 8 Mitglieder, 10 Termine
  Mindestens nötige Änderungen: 0, bester Termin: 4
  (gleich gute Termine: 4)
beispieldaten/praeferenzen3.txt: 14 Mitglieder, 20 Termine
  Mindestens nötige Änderungen: 7, bester Termin: 18
  (gleich gute Termine: 18)
beispieldaten/praeferenzen4.txt: 26 Mitglieder, 40 Termine
  Mindestens nötige Änderungen: 14, bester Termin: 22
  (gleich gute Termine: 22)
beispieldaten/praeferenzen5.txt: 50 Mitglieder, 80 Termine
  Mindestens nötige Änderungen: 34, bester Termin: 31
  (gleich gute Termine: 31, 56)


Das stimmt mit den Werten in den offiziellen Lösungshinweisen überein: 2 (Termin 6), 1 (Termin 2 oder 3), 0 (Termin 4), 7 (Termin 18),
14 (Termin 22) und 34 (Termin 31 oder 56). Bei `praeferenzen2.txt` gibt es einen Termin, der schon ohne Änderung allseits beliebt ist.

Ein Termin mit der geringsten Summe der Bewertungen oder den meisten grünen Einträgen ist **nicht** automatisch der beste: Es kommt nur darauf
an, ob jemand einen anderen Termin besser bewertet hat.

## Für Neugierige: Die Antwort überprüfen

Wir wollen sicher sein, dass der gefundene Termin nach den Änderungen wirklich allseits beliebt ist und dass wirklich so viele Änderungen nötig
waren. Dazu schreiben wir die Definition direkt als Funktion hin (ohne den Trick mit den besten Bewertungen):

- `ist_allseits_beliebt(tabelle, t)`: Für jedes Mitglied darf es keinen Termin geben, der besser bewertet ist als `t`.
- `aenderungen_fuer(tabelle, t)`: Liste der nötigen Änderungen `[Mitglied, alter Wert, neuer Wert]`, wir setzen jeweils auf die beste Bewertung des Mitglieds.

Dann wenden wir die Änderungen auf eine Kopie der Tabelle an (`zeile.copy()` macht eine Kopie der Zeile) und prüfen die Definition.

In [ ]:
def ist_allseits_beliebt(tabelle, t):
    for zeile in tabelle:
        for andere in range(len(zeile)):
            if zeile[andere] < zeile[t]:
                return False                # jemand findet einen anderen Termin besser
    return True


def aenderungen_fuer(tabelle, t):
    liste = []
    for i in range(len(tabelle)):
        beste = min(tabelle[i])
        if tabelle[i][t] > beste:
            liste.append([i, tabelle[i][t], beste])
    return liste


tabelle = lies_tabelle('beispieldaten/praeferenzen0.txt')
aenderungen, termine = bester_termin(tabelle)
t = termine[0]
print('Termin', t + 1, 'vorher allseits beliebt?', ist_allseits_beliebt(tabelle, t))

farben = ['grün', 'gelb', 'rot']
namen = ['Ada', 'Nancy', 'Niklaus', 'Grace', 'Edsger', 'Rósza']
neu = []
for zeile in tabelle:
    neu.append(zeile.copy())
for i, alt, besser in aenderungen_fuer(tabelle, t):
    print(f'  {namen[i]}: Termin {t + 1} von {farben[alt]} auf {farben[besser]}')
    neu[i][t] = besser
print('Termin', t + 1, 'nachher allseits beliebt?', ist_allseits_beliebt(neu, t))

Termin 6 vorher allseits beliebt? False
  Niklaus: Termin 6 von rot auf gelb
  Rósza: Termin 6 von gelb auf grün
Termin 6 nachher allseits beliebt? True


Die Änderungen sind etwas anders als im Beispiel der Aufgabe (dort wird bei Niklaus und Rósza *verschlechtert* statt *verbessert*),
aber es sind ebenfalls genau 2. Wir ändern immer den Eintrag des gewählten Termins, das reicht immer.

Zum Schluss prüfen wir für alle Beispiele, dass die Zahl der Änderungen zum gefundenen Termin passt und dass der Termin danach allseits beliebt ist:

In [ ]:
for nr in range(6):
    tabelle = lies_tabelle(f'beispieldaten/praeferenzen{nr}.txt')
    anzahl, termine = bester_termin(tabelle)
    t = termine[0]
    neu = []
    for zeile in tabelle:
        neu.append(zeile.copy())
    for i, alt, besser in aenderungen_fuer(tabelle, t):
        neu[i][t] = besser
    ok = len(aenderungen_fuer(tabelle, t)) == anzahl and ist_allseits_beliebt(neu, t)
    print(f'praeferenzen{nr}.txt: Termin {t + 1}, Änderungen: {anzahl}, danach allseits beliebt: {ok}')

praeferenzen0.txt: Termin 6, Änderungen: 2, danach allseits beliebt: True
praeferenzen1.txt: Termin 2, Änderungen: 1, danach allseits beliebt: True
praeferenzen2.txt: Termin 4, Änderungen: 0, danach allseits beliebt: True
praeferenzen3.txt: Termin 18, Änderungen: 7, danach allseits beliebt: True
praeferenzen4.txt: Termin 22, Änderungen: 14, danach allseits beliebt: True
praeferenzen5.txt: Termin 31, Änderungen: 34, danach allseits beliebt: True
